# LLM-based CartPole Reinforcement Learning Agent

This notebook demonstrates how to create and train a Reinforcement Learning agent that uses a Large Language Model (LLM) to make decisions in the CartPole environment.

## Setup

First, let's import the necessary libraries and set up our environment.

In [ ]:
!pip install isopro

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.5 MB/s eta 0:00:00
  Using cached tqdm-4.66.5-py3-none-any.whl (78 kB)
  Using cached gymnasium-0.29.1-py3-none-any.whl (953 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 36.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 19.0 MB/s eta 0:00:00
  Using cached rouge-1.0.1-py3-none-any.whl (13 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 7.1 MB/s eta 0:00:00
  Using cached nltk-3.9.1-py3-none-any.whl (1.5 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 55.9 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 53.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.1/376.1 kB 38.7 MB/s eta 0:00:00
  Using cached stable_baselines3-2.3.2-py3-none-any.whl (182 kB)
  Using cached torch-2.2.2-cp38-none-macosx_10_9_x86_64.whl (150.6 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.9/891.9 kB 4

In [1]:
import gymnasium as gym
from isopro.rl.rl_agent import RLAgent
from isopro.rl.rl_environment import LLMRLEnvironment
from stable_baselines3 import PPO
import numpy as np
import anthropic
import os
import logging
from typing import Optional, Dict, Any
from tqdm import tqdm
import json
from datetime import datetime
from llm_cartpole_wrapper import LLMCartPoleWrapper

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

: 

: 

## Create and Train the RL Agent

Now, let's create our RL agent and train it using the LLM-based CartPole environment.

In [ ]:
agent_prompt = """You are an AI trained to play the CartPole game. 
Your goal is to balance a pole on a moving cart for as long as possible. 
You will receive observations about the cart's position, velocity, pole angle, and angular velocity. 
Based on these, you should decide whether to move the cart left or right. 
Respond with 'Move left' or 'Move right' for each decision."""

env = LLMCartPoleWrapper(agent_prompt)
model = PPO("MlpPolicy", env, verbose=1)

logger.info("Starting training")
model.learn(total_timesteps=10000)
logger.info("Training completed")

## Test the Trained Agent

Now that we've trained our agent, let's test it for 2 episodes and see how it performs.

In [ ]:
test_episodes = 2
results = []

logger.info("Starting test episodes")
for episode in tqdm(range(test_episodes), desc="Test Episodes"):
    obs, _ = env.reset()
    done = False
    total_reward = 0
    episode_length = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        episode_length += 1
        done = terminated or truncated
    
    logger.info(f"Episode {episode + 1} completed. Total reward: {total_reward}, Length: {episode_length}")
    results.append({"episode": episode + 1, "total_reward": total_reward, "length": episode_length})

# Save results to file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = os.path.join(output_folder, f"cartpole_results_{timestamp}.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)
logger.info(f"Results saved to {output_file}")

# Print summary
average_reward = sum(r['total_reward'] for r in results) / len(results)
average_length = sum(r['length'] for r in results) / len(results)
logger.info(f"Test completed. Average reward: {average_reward:.2f}, Average length: {average_length:.2f}")

## Conclusion

In this notebook, we've demonstrated how to:

1. Set up an LLM-based wrapper for the CartPole environment
2. Train a reinforcement learning agent using this environment
3. Test the trained agent and collect performance metrics

This approach combines the decision-making capabilities of a large language model with the learning process of reinforcement learning, potentially leading to interesting and novel solutions to the CartPole problem.

Feel free to experiment with different prompts, training parameters, or even different environments to see how this approach can be applied in various scenarios!